# SEC EDGAR Financial Data Extraction for Fraud / Anomaly Detection
**Deliverable 1: Data extraction → secondary dataset**

This notebook pulls 5+ years of XBRL financial statement data from SEC EDGAR's free JSON APIs,
converts it into a clean **company × quarter** panel, and engineers features that are commonly
used to spot accounting anomalies (revenue growth shocks, expense-to-revenue shifts, receivables
outpacing sales, accruals, restatements, Beneish M-Score inputs).

**Pipeline**
1. Resolve tickers → CIK numbers (`company_tickers.json`)
2. Download company metadata (`/submissions`) and all XBRL facts (`/api/xbrl/companyfacts`)
3. Map heterogeneous US-GAAP tags to standard line items (with fallbacks for tag changes, e.g. ASC 606)
4. De-duplicate facts; use values **as first reported** (point-in-time) and record restatements (first-reported vs latest)
5. Convert year-to-date (YTD) figures into discrete quarters (Q2 = H1 − Q1, Q4 = FY − 9M, …)
6. Engineer quarterly and annual anomaly features
7. Validate and export CSVs to `data/processed/`

**Outputs**
| File | Grain | Purpose |
|---|---|---|
| `data/processed/quarterly_features.csv` | company × fiscal quarter | **Main secondary dataset** for Isolation Forest / Z-score |
| `data/processed/annual_features.csv` | company × fiscal year | Annual ratios + Beneish M-Score |
| `data/processed/facts_long.csv` | one row per XBRL fact | Audit trail back to each filing (accession number) |
| `data/processed/coverage_report.csv` | company | Data completeness summary |

> **Before running:** set the `SEC_USER_AGENT` environment variable (or edit `USER_AGENT` below) to your name and e-mail.
> The SEC blocks requests without one. Using the environment variable keeps your e-mail out of the repository.

In [ ]:
# Uncomment on a fresh environment
# %pip install pandas numpy requests

In [1]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

## 1. Configuration

In [2]:
# --- REQUIRED: the SEC asks every automated client to identify itself ("Name email") ---
USER_AGENT = os.environ.get("SEC_USER_AGENT", "Aayush aayushkanjani6@gmail.com")  # <-- set env var or edit

START_YEAR = 2014          # first fiscal year kept in the output (one extra year is used for lags)
END_YEAR = None            # None = through the latest filing

# If the notebook sits in a repo's notebooks/ folder, write to <repo>/data/processed; otherwise next to the notebook
_here = Path.cwd()
REPO_ROOT = _here.parent if _here.name == "notebooks" else _here
CACHE_DIR = REPO_ROOT / "edgar_cache"            # raw JSON cache (git-ignored); re-runs are instant and offline
OUTPUT_DIR = REPO_ROOT / "data" / "processed"    # CSV outputs
print("CSV outputs will be written to:", OUTPUT_DIR.resolve())
REQUEST_INTERVAL = 0.12           # seconds between requests (SEC fair-access limit is 10/second)

# Quarter / annual period-length windows in days. Wide enough for 52/53-week fiscal years
# and retailers with 12/16-week quarters.
Q_MIN, Q_MAX = 75, 125
A_MIN, A_MAX = 340, 380

VALID_FORMS = {"10-K", "10-Q", "10-K/A", "10-Q/A", "10-KT", "10-KT/A"}

CSV outputs will be written to: C:\Users\Asus\Downloads\h1\data\processed


### Company universe
* **Control group** – large, long-listed filers with no widely reported accounting enforcement in the period.
* **Reference group** – companies with publicly reported restatements or SEC accounting actions. Useful as
  weak labels to sanity-check the anomaly models in Deliverable 2. **Verify each against the SEC's
  Accounting and Auditing Enforcement Releases (AAER) and the company's own filings before using them as ground truth.**

Add any tickers you like. Companies that no longer trade (not in the ticker file) can be added by CIK in `EXTRA_CIKS`.

In [3]:
CONTROL_TICKERS = [
    "AAPL", "MSFT", "WMT", "KO", "PG", "JNJ", "PEP", "COST", "HD", "INTC",
    "CSCO", "ORCL", "NKE", "MCD", "CAT", "MMM", "HON", "TGT", "LOW", "ADBE",
]

REFERENCE_TICKERS = {
    "KHC":  "Restated 2016-2018 results in 2019 over supplier-cost accounting; SEC settlement 2021",
    "UAA":  "SEC settlement 2021 over undisclosed pull-forward of sales (2015-2016)",
    "SMCI": "SEC settlement 2020 over improper revenue recognition (FY2015-2017); auditor resigned 2024",
    "MDXG": "Revenue-recognition investigation; restated 2012-2016 results (filed 2020)",
    "PLUG": "Restated 2018-2020 financial statements in 2021",
    "MAT":  "Restated 2017-2018 interim results in 2019 (tax valuation allowance error)",
}

EXTRA_CIKS = {}   # e.g. {1234567: "Delisted Co - note"}

## 2. EDGAR access helpers (rate limiting, retries, disk cache)

In [4]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate"})
_last_request = [0.0]


def sec_get_json(url, cache_path=None, max_retries=5):
    """GET a JSON document from SEC with caching, throttling and exponential back-off.
    Returns None for 404 (e.g. a company with no XBRL facts)."""
    if cache_path is not None and cache_path.exists():
        return json.loads(cache_path.read_text())

    if "example.com" in USER_AGENT:
        raise ValueError("Set USER_AGENT to your real name and e-mail before downloading from SEC.")

    for attempt in range(max_retries):
        wait = REQUEST_INTERVAL - (time.time() - _last_request[0])
        if wait > 0:
            time.sleep(wait)
        _last_request[0] = time.time()
        try:
            r = SESSION.get(url, timeout=60)
        except requests.RequestException as exc:
            print(f"  network error ({exc}); retrying")
            time.sleep(2 ** attempt)
            continue
        if r.status_code == 200:
            data = r.json()
            if cache_path is not None:
                cache_path.parent.mkdir(parents=True, exist_ok=True)
                cache_path.write_text(json.dumps(data))
            return data
        if r.status_code == 404:
            return None
        if r.status_code in (403, 429, 500, 502, 503, 504):
            time.sleep(2 ** attempt + 1)
            continue
        r.raise_for_status()
    raise RuntimeError(f"Failed after {max_retries} attempts: {url}")

## 3. Resolve tickers → CIK and download raw data

In [5]:
def load_ticker_map():
    data = sec_get_json("https://www.sec.gov/files/company_tickers.json",
                        CACHE_DIR / "company_tickers.json")
    df = pd.DataFrame.from_dict(data, orient="index")
    df["cik"] = df["cik_str"].astype(int)
    df["ticker"] = df["ticker"].astype(str).str.upper()
    return df[["cik", "ticker", "title"]]


ticker_map = load_ticker_map()

universe_rows = []
for t in CONTROL_TICKERS:
    universe_rows.append({"ticker": t, "group": "control", "reference_note": ""})
for t, note in REFERENCE_TICKERS.items():
    universe_rows.append({"ticker": t, "group": "reference", "reference_note": note})
universe = pd.DataFrame(universe_rows).merge(ticker_map, on="ticker", how="left")

missing = universe[universe["cik"].isna()]["ticker"].tolist()
if missing:
    print("Tickers not found in SEC ticker file (add them via EXTRA_CIKS):", missing)
universe = universe.dropna(subset=["cik"])
universe["cik"] = universe["cik"].astype(int)

for cik, note in EXTRA_CIKS.items():
    universe.loc[len(universe)] = {"ticker": f"CIK{cik}", "group": "reference",
                                   "reference_note": note, "cik": int(cik), "title": ""}

# Share classes (e.g. UA / UAA) share one CIK -> keep one row per company
universe = universe.drop_duplicates("cik").reset_index(drop=True)
print(f"{len(universe)} companies in universe")
universe.head(30)

26 companies in universe


,ticker,group,reference_note,cik,title
0,AAPL,control,,320193,Apple Inc.
1,MSFT,control,,789019,MICROSOFT CORP
2,WMT,control,,104169,Walmart Inc.
3,KO,control,,21344,COCA COLA CO
4,PG,control,,80424,PROCTER & GAMBLE Co
5,JNJ,control,,200406,JOHNSON & JOHNSON
6,PEP,control,,77476,PEPSICO INC
7,COST,control,,909832,COSTCO WHOLESALE CORP /NEW
8,HD,control,,354950,"HOME DEPOT, INC."
9,INTC,control,,50863,INTEL CORP


In [6]:
def fetch_company(cik):
    cik10 = f"{cik:010d}"
    subs = sec_get_json(f"https://data.sec.gov/submissions/CIK{cik10}.json",
                        CACHE_DIR / "submissions" / f"CIK{cik10}.json")
    facts = sec_get_json(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik10}.json",
                         CACHE_DIR / "companyfacts" / f"CIK{cik10}.json")
    return subs, facts


raw = {}
meta_rows = []
for i, row in universe.iterrows():
    subs, facts = fetch_company(row.cik)
    raw[row.cik] = facts
    subs = subs or {}
    meta_rows.append({
        "cik": row.cik,
        "ticker": row.ticker,
        "company": subs.get("name") or row.title,
        "sic": subs.get("sic"),
        "sic_description": subs.get("sicDescription"),
        "fiscal_year_end": subs.get("fiscalYearEnd"),
        "state_of_incorporation": subs.get("stateOfIncorporation"),
        "group": row.group,
        "reference_note": row.reference_note,
    })
    print(f"[{i + 1}/{len(universe)}] {row.ticker:<6} facts={'yes' if facts else 'NO'}")

company_meta = pd.DataFrame(meta_rows)
company_meta.head()

[1/26] AAPL   facts=yes
[2/26] MSFT   facts=yes
[3/26] WMT    facts=yes
[4/26] KO     facts=yes
[5/26] PG     facts=yes
[6/26] JNJ    facts=yes
[7/26] PEP    facts=yes
[8/26] COST   facts=yes
[9/26] HD     facts=yes
[10/26] INTC   facts=yes
[11/26] CSCO   facts=yes
[12/26] ORCL   facts=yes
[13/26] NKE    facts=yes
[14/26] MCD    facts=yes
[15/26] CAT    facts=yes
[16/26] MMM    facts=yes
[17/26] HON    facts=yes
[18/26] TGT    facts=yes
[19/26] LOW    facts=yes
[20/26] ADBE   facts=yes
[21/26] KHC    facts=yes
[22/26] UAA    facts=yes
[23/26] SMCI   facts=yes
[24/26] MDXG   facts=yes
[25/26] PLUG   facts=yes
[26/26] MAT    facts=yes


,cik,ticker,company,sic,sic_description,fiscal_year_end,state_of_incorporation,group,reference_note
0,320193,AAPL,Apple Inc.,3571,Electronic Computers,0926,CA,control,
1,789019,MSFT,MICROSOFT CORP,7372,Services-Prepackaged Software,0630,WA,control,
2,104169,WMT,Walmart Inc.,5331,Retail-Variety Stores,0131,DE,control,
3,21344,KO,COCA COLA CO,2080,Beverages,1231,DE,control,
4,80424,PG,PROCTER & GAMBLE Co,2840,"Soap, Detergents, Cleang Preparations, Perfume...",0630,OH,control,


## 4. Map US-GAAP tags to standard line items
Companies use different tags for the same concept, and tags change over time (e.g. `SalesRevenueNet` was
replaced by `RevenueFromContractWithCustomerExcludingAssessedTax` under ASC 606 in 2018).
Each line item has an **ordered** list of tags; for every period the highest-priority tag with a value wins,
and the tag used is recorded so tag switches can be excluded as a source of false anomalies.

In [7]:
CONCEPTS = {
    # ---------- income statement / cash flow (duration facts) ----------
    "revenue": ("duration", [
        "Revenues",
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "RevenueFromContractWithCustomerIncludingAssessedTax",
        "SalesRevenueNet",
        "SalesRevenueGoodsNet",
        "SalesRevenueServicesNet",
        "RevenuesNetOfInterestExpense",
    ]),
    "cost_of_revenue": ("duration", [
        "CostOfRevenue",
        "CostOfGoodsAndServicesSold",
        "CostOfGoodsSold",
        "CostOfGoodsAndServiceExcludingDepreciationDepletionAndAmortization",
        "CostOfServices",
    ]),
    "gross_profit": ("duration", ["GrossProfit"]),
    "sga": ("duration", ["SellingGeneralAndAdministrativeExpense"]),
    "rnd": ("duration", ["ResearchAndDevelopmentExpense",
                         "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost"]),
    "operating_expenses": ("duration", ["OperatingExpenses"]),
    "costs_and_expenses": ("duration", ["CostsAndExpenses"]),
    "operating_income": ("duration", ["OperatingIncomeLoss"]),
    "net_income": ("duration", ["NetIncomeLoss", "ProfitLoss",
                                "NetIncomeLossAvailableToCommonStockholdersBasic"]),
    "depreciation": ("duration", ["DepreciationDepletionAndAmortization",
                                  "DepreciationAndAmortization",
                                  "DepreciationAmortizationAndAccretionNet",
                                  "Depreciation"]),
    "cfo": ("duration", ["NetCashProvidedByUsedInOperatingActivities",
                         "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"]),
    # ---------- balance sheet (instant facts) ----------
    "accounts_receivable": ("instant", ["AccountsReceivableNetCurrent", "ReceivablesNetCurrent",
                                    "AccountsNotesAndLoansReceivableNetCurrent",
                                    "AccountsAndOtherReceivablesNetCurrent"]),
    "inventory": ("instant", ["InventoryNet"]),
    "current_assets": ("instant", ["AssetsCurrent"]),
    "total_assets": ("instant", ["Assets"]),
    "ppe_net": ("instant", ["PropertyPlantAndEquipmentNet"]),
    "current_liabilities": ("instant", ["LiabilitiesCurrent"]),
    "total_liabilities": ("instant", ["Liabilities"]),
    "liabilities_and_equity": ("instant", ["LiabilitiesAndStockholdersEquity"]),
    "long_term_debt": ("instant", ["LongTermDebtNoncurrent", "LongTermDebt"]),
    "equity": ("instant", ["StockholdersEquity",
                           "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"]),
    "cash": ("instant", ["CashAndCashEquivalentsAtCarryingValue",
                         "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents"]),
    "deferred_revenue": ("instant", ["ContractWithCustomerLiabilityCurrent", "DeferredRevenueCurrent"]),
}
DURATION_ITEMS = [k for k, (kind, _) in CONCEPTS.items() if kind == "duration"]
INSTANT_ITEMS = [k for k, (kind, _) in CONCEPTS.items() if kind == "instant"]

## 5. Flatten XBRL JSON → long fact table, de-duplicate, detect restatements
The same period is reported many times (in its own 10-Q, then as a comparative in later filings).
We keep the **latest filed** value (it reflects any restatement) and also keep the **first reported**
value. A difference between the two is a restatement or reclassification — a classic red flag.

In [8]:
# Guard: the download cell in Section 3 must have run successfully first
if "raw" not in globals() or not raw:
    raise RuntimeError("No downloaded data found. Restart the kernel and use 'Run All'; "
                       "if an earlier cell failed (e.g. USER_AGENT not set, HTTP 403), fix that first.")
if not any(raw.values()):
    raise RuntimeError("Downloads returned no XBRL facts for any company - check USER_AGENT and your internet connection.")


def facts_to_long(facts_json, cik):
    rows = []
    usgaap = (facts_json or {}).get("facts", {}).get("us-gaap", {})
    for item, (kind, tags) in CONCEPTS.items():
        for priority, tag in enumerate(tags):
            node = usgaap.get(tag)
            if not node:
                continue
            for obs in node.get("units", {}).get("USD", []):
                if obs.get("form") not in VALID_FORMS:
                    continue
                rows.append({
                    "cik": cik, "item": item, "kind": kind, "tag": tag, "tag_priority": priority,
                    "start": obs.get("start"), "end": obs.get("end"), "val": obs.get("val"),
                    "form": obs.get("form"), "filed": obs.get("filed"), "accn": obs.get("accn"),
                    "fy": obs.get("fy"), "fp": obs.get("fp"),
                })
    return rows


all_rows = []
for cik, fj in raw.items():
    all_rows.extend(facts_to_long(fj, cik))
facts_long = pd.DataFrame(all_rows)
for c in ("start", "end", "filed"):
    facts_long[c] = pd.to_datetime(facts_long[c])
facts_long["val"] = pd.to_numeric(facts_long["val"], errors="coerce")
facts_long["duration_days"] = (facts_long["end"] - facts_long["start"]).dt.days
facts_long = facts_long.dropna(subset=["end", "val"])
print(f"{len(facts_long):,} raw facts")


def dedupe_with_restatements(df, rel_tol=0.005):
    """One row per (cik, item, tag, start, end): latest value, first-reported value, restatement flag."""
    key = ["cik", "item", "kind", "tag", "tag_priority", "start", "end"]
    df = df.sort_values("filed", kind="mergesort")
    g = df.groupby(key, dropna=False, sort=False)
    out = g.agg(
        val=("val", "last"),
        first_val=("val", "first"),
        first_filed=("filed", "first"),
        last_filed=("filed", "last"),
        last_form=("form", "last"),
        last_accn=("accn", "last"),
        n_reports=("val", "size"),
    ).reset_index()
    out["duration_days"] = (out["end"] - out["start"]).dt.days
    diff = (out["val"] - out["first_val"]).abs()
    denom = out["first_val"].abs().replace(0, np.nan)
    out["restatement_pct"] = (out["val"] - out["first_val"]) / denom
    out["restated"] = (diff > rel_tol * out["first_val"].abs()) & (diff > 0)
    out["amended"] = out["last_form"].str.endswith("/A")
    return out


facts = dedupe_with_restatements(facts_long)
print(f"{len(facts):,} unique facts; {facts['restated'].sum():,} restated")

106,153 raw facts
47,514 unique facts; 1,993 restated


## 6. Build discrete quarters from YTD / annual figures
10-Qs often report cash-flow (and sometimes income-statement) items only year-to-date, and 10-Ks
report only the full year, so Q4 is never filed directly. Discrete quarters are derived in order of preference:
1. **reported_3m** – a 3-month value filed directly
2. **ytd_diff** – consecutive YTD values with the same start date (Q2 = 6M − 3M, Q4 = 12M − 9M)
3. **annual_minus_3q** – full year minus the three known quarters

**All values are "as first reported" (point-in-time).** The 12-month and 9-month figures were first filed within
months of each other, on the same accounting basis. Differencing the *latest* values instead mixes vintages: after a
spin-off or discontinued operation, the full year is recast in a 10-K years later but the 9-month figure never is,
which produces a meaningless Q4. Point-in-time data is also what investors saw, so it avoids look-ahead bias.
The latest (restated/recast) value is kept separately as `latest_val` where it can be derived consistently.

In [9]:
ONE_DAY = pd.Timedelta(days=1)


def quarterize_series(d):
    """d: de-duplicated duration facts for one (cik, item, tag). Returns {end: record}."""
    q = {}
    for r in d[d["duration_days"].between(Q_MIN, Q_MAX)].itertuples():
        q[r.end] = dict(start=r.start, end=r.end, val=r.first_val, latest_val=r.val,
                        restated=bool(r.restated), amended=bool(r.amended), method="reported_3m")

    for _, g in d.groupby("start"):
        g = g.sort_values("end")
        recs = list(g.itertuples())
        for prev, cur in zip(recs[:-1], recs[1:]):
            gap = (cur.end - prev.end).days
            if Q_MIN <= gap <= Q_MAX and cur.end not in q:
                # latest values can only be differenced if both were revised together (same vintage)
                same_vintage = bool(cur.restated) == bool(prev.restated)
                q[cur.end] = dict(start=prev.end + ONE_DAY, end=cur.end,
                                  val=cur.first_val - prev.first_val,
                                  latest_val=(cur.val - prev.val) if same_vintage else np.nan,
                                  restated=bool(cur.restated),
                                  amended=bool(cur.amended or prev.amended), method="ytd_diff")

    for r in d[d["duration_days"].between(A_MIN, A_MAX)].itertuples():
        if r.end in q:
            continue
        inner = [v for e, v in q.items()
                 if e < r.end and v["start"] >= r.start - pd.Timedelta(days=7)]
        if len(inner) == 3:
            same_vintage = all(v["restated"] == bool(r.restated) and pd.notna(v["latest_val"]) for v in inner)
            q[r.end] = dict(start=max(v["end"] for v in inner) + ONE_DAY, end=r.end,
                            val=r.first_val - sum(v["val"] for v in inner),
                            latest_val=(r.val - sum(v["latest_val"] for v in inner)) if same_vintage else np.nan,
                            restated=bool(r.restated),
                            amended=bool(r.amended), method="annual_minus_3q")
    return q


def build_quarterly_items(facts):
    recs = []
    dur = facts[facts["kind"] == "duration"]
    for (cik, item, tag, prio), d in dur.groupby(["cik", "item", "tag", "tag_priority"]):
        for end, v in quarterize_series(d).items():
            recs.append({"cik": cik, "item": item, "tag": tag, "tag_priority": prio, **v})
    qi = pd.DataFrame(recs)
    # highest-priority tag per (cik, item, quarter end)
    qi = qi.sort_values("tag_priority").drop_duplicates(["cik", "item", "end"], keep="first")
    return qi


q_items = build_quarterly_items(facts)
q_items["method"].value_counts()

method
reported_3m        10522
ytd_diff            4349
annual_minus_3q        1
Name: count, dtype: int64

In [10]:
def pick_by_priority(df, keys):
    return df.sort_values("tag_priority").drop_duplicates(keys, keep="first")


# ---- wide quarterly panel of duration items ----
q_wide = q_items.pivot_table(index=["cik", "end"], columns="item", values="val", aggfunc="first")
q_wide = q_wide.reindex(columns=DURATION_ITEMS)
q_start = q_items.groupby(["cik", "end"])["start"].min().rename("period_start")
panel = q_wide.join(q_start).reset_index().rename(columns={"end": "period_end"})

# provenance / restatement info for revenue specifically, plus counts across all items
rev = q_items[q_items["item"] == "revenue"].set_index(["cik", "end"])
panel = panel.set_index(["cik", "period_end"])
panel["revenue_tag"] = rev["tag"]
panel["revenue_method"] = rev["method"]
panel["revenue_latest"] = rev["latest_val"]
restate_counts = q_items.groupby(["cik", "end"]).agg(
    n_items_restated=("restated", "sum"), any_amended=("amended", "any"))
restate_counts.index.names = ["cik", "period_end"]
panel = panel.join(restate_counts).reset_index()

# ---- balance sheet items at each quarter end (exact date, else nearest within 10 days) ----
inst = pick_by_priority(facts[facts["kind"] == "instant"], ["cik", "item", "end"])
inst_wide = inst.pivot_table(index=["cik", "end"], columns="item", values="first_val", aggfunc="first")
inst_wide = inst_wide.reindex(columns=INSTANT_ITEMS).reset_index().rename(columns={"end": "bs_date"})
inst_restated = inst.groupby(["cik", "end"])["restated"].sum().rename("n_bs_items_restated").reset_index()
inst_wide = inst_wide.merge(inst_restated.rename(columns={"end": "bs_date"}), on=["cik", "bs_date"], how="left")

panel = panel.sort_values("period_end")
inst_wide = inst_wide.sort_values("bs_date")
panel = pd.merge_asof(panel, inst_wide, left_on="period_end", right_on="bs_date", by="cik",
                      direction="nearest", tolerance=pd.Timedelta(days=10))
panel = panel.sort_values(["cik", "period_end"]).reset_index(drop=True)

# ---- fill gaps using accounting identities ----
m = panel["gross_profit"].isna()
panel.loc[m, "gross_profit"] = panel["revenue"] - panel["cost_of_revenue"]
m = panel["cost_of_revenue"].isna()
panel.loc[m, "cost_of_revenue"] = panel["revenue"] - panel["gross_profit"]
m = panel["total_liabilities"].isna()
panel.loc[m, "total_liabilities"] = panel["liabilities_and_equity"] - panel["equity"]
print(panel.shape)
panel.head()

(1844, 33)


,cik,period_end,revenue,cost_of_revenue,gross_profit,sga,rnd,operating_expenses,costs_and_expenses,operating_income,net_income,depreciation,cfo,period_start,revenue_tag,revenue_method,revenue_latest,n_items_restated,any_amended,bs_date,accounts_receivable,inventory,current_assets,total_assets,ppe_net,current_liabilities,total_liabilities,liabilities_and_equity,long_term_debt,equity,cash,deferred_revenue,n_bs_items_restated
0,18230,2008-06-30,1.362400e+10,-1.003600e+10,2.366000e+10,-1.074000e+09,-415000000.0,1.209900e+10,NaN,1.525000e+09,1.106000e+09,NaN,NaN,2008-04-01,Revenues,reported_3m,1.362400e+10,0,True,2008-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.583000e+09,7.820000e+08,NaN,0.0
1,18230,2008-09-30,1.298100e+10,9.704000e+09,3.277000e+09,1.061000e+09,437000000.0,1.180800e+10,NaN,1.173000e+09,8.680000e+08,501000000.0,1.375000e+09,2008-07-01,Revenues,reported_3m,1.298100e+10,0,True,2008-09-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.760000e+09,2.138000e+09,NaN,0.0
2,18230,2008-12-31,1.292300e+10,1.006600e+10,2.857000e+09,1.305000e+09,507000000.0,1.246600e+10,NaN,4.570000e+08,6.610000e+08,527000000.0,1.517000e+09,2008-10-01,Revenues,ytd_diff,1.292300e+10,1,False,2008-12-31,9.397000e+09,8.781000e+09,3.163300e+10,6.778200e+10,1.252400e+10,2.606900e+10,6.106800e+10,6.778200e+10,2.283400e+10,6.190000e+09,2.736000e+09,NaN,2.0
3,18230,2009-03-31,9.225000e+09,7.027000e+09,1.483000e+09,8.820000e+08,388000000.0,NaN,9.400000e+09,-1.750000e+08,-1.120000e+08,534000000.0,8.950000e+08,2008-12-30,Revenues,reported_3m,9.225000e+09,0,False,2009-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.336000e+09,3.566000e+09,NaN,0.0
4,18230,2009-06-30,7.975000e+09,5.752000e+09,1.502000e+09,-9.140000e+08,-351000000.0,7.628000e+09,NaN,3.470000e+08,3.710000e+08,538000000.0,1.908000e+09,2009-03-31,Revenues,reported_3m,7.975000e+09,2,True,2009-06-30,6.534000e+09,7.160000e+09,2.738300e+10,6.209200e+10,1.222600e+10,1.949000e+10,5.423600e+10,6.209200e+10,NaN,7.375000e+09,3.991000e+09,NaN,0.0


## 7. Fiscal calendar
Fiscal year/quarter are derived from each company's fiscal-year-end (from `/submissions`), shifted 14 days
so that 52/53-week years ending in the first days of a month are labelled correctly.

In [11]:
def fiscal_labels(period_end, fye):
    shifted = period_end - pd.Timedelta(days=14)
    try:
        fye_date = pd.Timestamp(2001, int(str(fye)[:2]), int(str(fye)[2:4])) - pd.Timedelta(days=14)
        fye_month = fye_date.month
    except (ValueError, TypeError):
        fye_month = 12
    fq = ((shifted.month - fye_month - 1) % 12) // 3 + 1
    fy = shifted.year + (1 if shifted.month > fye_month else 0)
    return fy, fq


panel = panel.merge(company_meta, on="cik", how="left")
labels = [fiscal_labels(pe, fye) for pe, fye in zip(panel["period_end"], panel["fiscal_year_end"])]
panel["fiscal_year"] = [l[0] for l in labels]
panel["fiscal_quarter"] = [l[1] for l in labels]
panel["period_days"] = (panel["period_end"] - panel["period_start"]).dt.days + 1
panel["calendar_quarter"] = (panel["period_end"] - pd.Timedelta(days=14)).dt.to_period("Q").astype(str)

### Sanity check against full-year totals
Some filers occasionally tag a full-year number with a 3-month context (a source-data error). Any quarter whose
revenue exceeds 90% of that fiscal year's revenue is set to missing and flagged, so it cannot distort growth rates
or the trailing Z-scores of the following quarters. Years whose four quarters do not add up to the annual figure
(within 2%) are flagged for review.

In [12]:
ann_rev = pick_by_priority(facts[(facts["item"] == "revenue") & facts["duration_days"].between(A_MIN, A_MAX)],
                           ["cik", "end"])[["cik", "end", "first_val"]]
ann_rev = ann_rev.merge(company_meta[["cik", "fiscal_year_end"]], on="cik", how="left")
ann_rev["fiscal_year"] = [fiscal_labels(e, fye)[0] for e, fye in zip(ann_rev["end"], ann_rev["fiscal_year_end"])]
ann_rev = (ann_rev.sort_values("end").groupby(["cik", "fiscal_year"])["first_val"].last()
           .rename("_annual_revenue").reset_index())
panel = panel.merge(ann_rev, on=["cik", "fiscal_year"], how="left")

panel["dq_quarter_looks_annual"] = (panel["_annual_revenue"] > 0) & (panel["revenue"] > 0.9 * panel["_annual_revenue"])
print(f"Quarters holding a full-year value (revenue set to missing): {panel['dq_quarter_looks_annual'].sum()}")
panel.loc[panel["dq_quarter_looks_annual"], ["revenue", "revenue_latest"]] = np.nan

_g = panel.groupby(["cik", "fiscal_year"])["revenue"]
_qsum, _qn = _g.transform("sum"), _g.transform("count")
panel["dq_quarter_sum_mismatch"] = (_qn == 4) & ((_qsum - panel["_annual_revenue"]).abs()
                                                 > 0.02 * panel["_annual_revenue"].abs())
panel = panel.drop(columns="_annual_revenue")

Quarters holding a full-year value (revenue set to missing): 5


## 8. Feature engineering (quarterly)
Lags are matched on **dates**, not row positions, so a missing quarter never produces a wrong growth rate.
* QoQ lag = previous quarter ending 75–125 days earlier
* YoY lag = same quarter a year earlier (340–390 days) — removes seasonality

In [13]:
def safe_div(a, b):
    b = b.where(b != 0)
    return a / b


def add_lag(df, cols, min_days, max_days, suffix):
    """For each row, attach values from the same company whose period_end is min..max days earlier."""
    left = df[["cik", "period_end"]].copy()
    left["target"] = left["period_end"] - pd.Timedelta(days=(min_days + max_days) // 2)
    right = df[["cik", "period_end"] + cols].rename(
        columns={"period_end": "lag_end", **{c: f"{c}{suffix}" for c in cols}})
    left = left.sort_values("target")
    right = right.sort_values("lag_end")
    merged = pd.merge_asof(left, right, left_on="target", right_on="lag_end", by="cik",
                           direction="nearest", tolerance=pd.Timedelta(days=(max_days - min_days) // 2))
    merged = merged.set_index(["cik", "period_end"]).drop(columns=["target", "lag_end"])
    return df.join(merged, on=["cik", "period_end"])


LAG_COLS = ["revenue", "cost_of_revenue", "gross_profit", "sga", "operating_income", "net_income",
            "cfo", "accounts_receivable", "inventory", "total_assets", "deferred_revenue",
            "operating_expenses"]
panel = add_lag(panel, LAG_COLS, Q_MIN, Q_MAX, "_prev_q")
panel = add_lag(panel, LAG_COLS, 340, 390, "_prev_y")

f = panel  # alias for readability
f["days"] = f["period_days"].fillna(91)

# --- growth ---
f["revenue_qoq_growth"] = safe_div(f["revenue"], f["revenue_prev_q"]) - 1
f["revenue_yoy_growth"] = safe_div(f["revenue"], f["revenue_prev_y"]) - 1
f["cogs_yoy_growth"] = safe_div(f["cost_of_revenue"], f["cost_of_revenue_prev_y"]) - 1
f["sga_yoy_growth"] = safe_div(f["sga"], f["sga_prev_y"]) - 1
f["net_income_yoy_growth"] = safe_div(f["net_income"] - f["net_income_prev_y"], f["net_income_prev_y"].abs())
f["ar_yoy_growth"] = safe_div(f["accounts_receivable"], f["accounts_receivable_prev_y"]) - 1
f["inventory_yoy_growth"] = safe_div(f["inventory"], f["inventory_prev_y"]) - 1
f["deferred_revenue_yoy_growth"] = safe_div(f["deferred_revenue"], f["deferred_revenue_prev_y"]) - 1

# --- expense-to-revenue ratios & margins ---
f["cogs_to_revenue"] = safe_div(f["cost_of_revenue"], f["revenue"])
f["sga_to_revenue"] = safe_div(f["sga"], f["revenue"])
f["rnd_to_revenue"] = safe_div(f["rnd"], f["revenue"])
f["opex_to_revenue"] = safe_div(f["operating_expenses"], f["revenue"])
# all operating costs, robust to missing expense tags: (revenue - operating income) / revenue
f["total_expense_to_revenue"] = safe_div(f["revenue"] - f["operating_income"], f["revenue"])
f["gross_margin"] = safe_div(f["gross_profit"], f["revenue"])
f["operating_margin"] = safe_div(f["operating_income"], f["revenue"])
f["net_margin"] = safe_div(f["net_income"], f["revenue"])

gm_prev_y = safe_div(f["gross_profit_prev_y"], f["revenue_prev_y"])
om_prev_y = safe_div(f["operating_income_prev_y"], f["revenue_prev_y"])
ex_prev_y = safe_div(f["revenue_prev_y"] - f["operating_income_prev_y"], f["revenue_prev_y"])
f["gross_margin_yoy_change"] = f["gross_margin"] - gm_prev_y
f["operating_margin_yoy_change"] = f["operating_margin"] - om_prev_y
f["expense_ratio_yoy_change"] = f["total_expense_to_revenue"] - ex_prev_y

# --- working capital & earnings-quality red flags ---
f["dso"] = safe_div(f["accounts_receivable"], f["revenue"]) * f["days"]          # days sales outstanding
f["dio"] = safe_div(f["inventory"], f["cost_of_revenue"]) * f["days"]           # days inventory outstanding
f["ar_growth_minus_revenue_growth"] = f["ar_yoy_growth"] - f["revenue_yoy_growth"]   # channel-stuffing signal
f["inventory_growth_minus_revenue_growth"] = f["inventory_yoy_growth"] - f["revenue_yoy_growth"]
f["sga_growth_minus_revenue_growth"] = f["sga_yoy_growth"] - f["revenue_yoy_growth"]
avg_assets = f[["total_assets", "total_assets_prev_q"]].mean(axis=1)
f["accruals_to_assets"] = safe_div(f["net_income"] - f["cfo"], avg_assets)       # Sloan accruals
f["cfo_to_net_income"] = safe_div(f["cfo"], f["net_income"].abs())
f["leverage"] = safe_div(f["total_liabilities"], f["total_assets"])
f["current_ratio"] = safe_div(f["current_assets"], f["current_liabilities"])
f["revenue_restatement_pct"] = safe_div(f["revenue_latest"] - f["revenue"], f["revenue"].abs())

# tag switches are a common cause of artificial jumps -> flag them so they can be filtered
f = f.sort_values(["cik", "period_end"])
f["revenue_tag_changed"] = (f.groupby("cik")["revenue_tag"].shift(1) != f["revenue_tag"]) & \
                           f.groupby("cik")["revenue_tag"].shift(1).notna()
panel = f

### Unusual jumps / drops (trailing Z-scores)
For each company and metric: `z = (x_t − mean(x_{t−8..t−1})) / std(x_{t−8..t−1})`.
Only **past** quarters are used, so there is no look-ahead bias. A robust (median/MAD) version is also
included because a single extreme quarter inflates the ordinary standard deviation.
These columns are features; the anomaly models themselves belong to Deliverable 2.

In [14]:
Z_METRICS = ["revenue_qoq_growth", "revenue_yoy_growth", "total_expense_to_revenue", "gross_margin",
             "operating_margin", "dso", "accruals_to_assets", "ar_growth_minus_revenue_growth"]
WINDOW, MIN_PERIODS = 8, 4


def _floor(scale, center):
    # near-constant history -> undefined z instead of an astronomically large one
    # (all Z_METRICS are ratios or growth rates, so an absolute floor of 1e-4 is negligible in practice)
    return scale.where(scale > np.maximum(1e-4, 1e-3 * center.abs()))


def trailing_z(s):
    past = s.shift(1)
    mu = past.rolling(WINDOW, min_periods=MIN_PERIODS).mean()
    sd = past.rolling(WINDOW, min_periods=MIN_PERIODS).std()
    return (s - mu) / _floor(sd, mu)


def trailing_robust_z(s):
    # note: the MAD here is taken around the current window median (an approximation, fast and stable)
    past = s.shift(1)
    med = past.rolling(WINDOW, min_periods=MIN_PERIODS).median()
    mad = (past - med).abs().rolling(WINDOW, min_periods=MIN_PERIODS).median()
    return 0.6745 * (s - med) / _floor(mad, med)


panel = panel.sort_values(["cik", "period_end"]).reset_index(drop=True)
for col in Z_METRICS:
    panel[f"z_{col}"] = panel.groupby("cik")[col].transform(trailing_z)
    panel[f"rz_{col}"] = panel.groupby("cik")[col].transform(trailing_robust_z)

z_cols = [f"z_{c}" for c in Z_METRICS]
panel["n_abs_z_gt_3"] = (panel[z_cols].abs() > 3).sum(axis=1)
panel["max_abs_z"] = panel[z_cols].abs().max(axis=1)

## 9. Annual dataset with Beneish M-Score
Built from 10-K full-year figures (duration 340–380 days) and fiscal-year-end balances.
M-Score (8-variable): `−4.84 + 0.920·DSRI + 0.528·GMI + 0.404·AQI + 0.892·SGI + 0.115·DEPI − 0.172·SGAI + 4.679·TATA − 0.327·LVGI`.
Scores above about −1.78 are conventionally read as a higher likelihood of earnings manipulation.

In [15]:
ann = facts[(facts["kind"] == "duration") & facts["duration_days"].between(A_MIN, A_MAX)]
ann = pick_by_priority(ann, ["cik", "item", "end"])
a_wide = ann.pivot_table(index=["cik", "end"], columns="item", values="first_val", aggfunc="first")
a_wide = a_wide.reindex(columns=DURATION_ITEMS)
a_restated = ann.groupby(["cik", "end"])["restated"].sum().rename("n_items_restated")
annual = a_wide.join(a_restated).reset_index().rename(columns={"end": "period_end"}).sort_values("period_end")
annual = pd.merge_asof(annual, inst_wide, left_on="period_end", right_on="bs_date", by="cik",
                       direction="nearest", tolerance=pd.Timedelta(days=10))
annual = annual.sort_values(["cik", "period_end"]).reset_index(drop=True)

m = annual["gross_profit"].isna()
annual.loc[m, "gross_profit"] = annual["revenue"] - annual["cost_of_revenue"]
m = annual["cost_of_revenue"].isna()
annual.loc[m, "cost_of_revenue"] = annual["revenue"] - annual["gross_profit"]
m = annual["total_liabilities"].isna()
annual.loc[m, "total_liabilities"] = annual["liabilities_and_equity"] - annual["equity"]

annual = annual.merge(company_meta, on="cik", how="left")
annual["fiscal_year"] = [fiscal_labels(pe, fye)[0] for pe, fye in zip(annual["period_end"], annual["fiscal_year_end"])]

BENEISH_INPUTS = ["revenue", "cost_of_revenue", "gross_profit", "sga", "depreciation", "net_income", "cfo",
                  "accounts_receivable", "current_assets", "ppe_net", "total_assets",
                  "current_liabilities", "long_term_debt", "operating_income"]
annual = add_lag(annual, BENEISH_INPUTS, 340, 390, "_prev_y")
a = annual

a["revenue_yoy_growth"] = safe_div(a["revenue"], a["revenue_prev_y"]) - 1
a["gross_margin"] = safe_div(a["gross_profit"], a["revenue"])
a["operating_margin"] = safe_div(a["operating_income"], a["revenue"])
a["net_margin"] = safe_div(a["net_income"], a["revenue"])
a["total_expense_to_revenue"] = safe_div(a["revenue"] - a["operating_income"], a["revenue"])
a["dso"] = safe_div(a["accounts_receivable"], a["revenue"]) * 365

gm_t = safe_div(a["revenue"] - a["cost_of_revenue"], a["revenue"])
gm_p = safe_div(a["revenue_prev_y"] - a["cost_of_revenue_prev_y"], a["revenue_prev_y"])
aq_t = 1 - safe_div(a["current_assets"] + a["ppe_net"], a["total_assets"])
aq_p = 1 - safe_div(a["current_assets_prev_y"] + a["ppe_net_prev_y"], a["total_assets_prev_y"])
dep_t = safe_div(a["depreciation"], a["depreciation"] + a["ppe_net"])
dep_p = safe_div(a["depreciation_prev_y"], a["depreciation_prev_y"] + a["ppe_net_prev_y"])
lv_t = safe_div(a["current_liabilities"] + a["long_term_debt"].fillna(0), a["total_assets"])
lv_p = safe_div(a["current_liabilities_prev_y"] + a["long_term_debt_prev_y"].fillna(0), a["total_assets_prev_y"])

a["beneish_dsri"] = safe_div(safe_div(a["accounts_receivable"], a["revenue"]),
                             safe_div(a["accounts_receivable_prev_y"], a["revenue_prev_y"]))
a["beneish_gmi"] = safe_div(gm_p, gm_t)
a["beneish_aqi"] = safe_div(aq_t, aq_p)
a["beneish_sgi"] = safe_div(a["revenue"], a["revenue_prev_y"])
a["beneish_depi"] = safe_div(dep_p, dep_t)
a["beneish_sgai"] = safe_div(safe_div(a["sga"], a["revenue"]), safe_div(a["sga_prev_y"], a["revenue_prev_y"]))
a["beneish_lvgi"] = safe_div(lv_t, lv_p)
a["beneish_tata"] = safe_div(a["net_income"] - a["cfo"], a["total_assets"])
a["beneish_m_score"] = (-4.84 + 0.920 * a["beneish_dsri"] + 0.528 * a["beneish_gmi"]
                        + 0.404 * a["beneish_aqi"] + 0.892 * a["beneish_sgi"]
                        + 0.115 * a["beneish_depi"] - 0.172 * a["beneish_sgai"]
                        + 4.679 * a["beneish_tata"] - 0.327 * a["beneish_lvgi"])
a["beneish_flag"] = a["beneish_m_score"] > -1.78
annual = a

## 10. Validation checks

In [16]:
# (a) four derived quarters should add up to the reported full year
qsum = (panel.dropna(subset=["revenue"])
        .groupby(["cik", "fiscal_year"])
        .agg(q_revenue_sum=("revenue", "sum"), n_q=("revenue", "size")).reset_index())
chk = qsum[qsum["n_q"] == 4].merge(annual[["cik", "fiscal_year", "revenue"]], on=["cik", "fiscal_year"])
chk["abs_pct_diff"] = (chk["q_revenue_sum"] - chk["revenue"]).abs() / chk["revenue"].abs()
chk_in = chk[chk["fiscal_year"] >= START_YEAR]
print(f"Quarter-sum vs annual revenue (all years incl. early XBRL): {len(chk)} company-years, "
      f"{(chk['abs_pct_diff'] > 0.01).mean():.1%} differ by >1%")
print(f"Quarter-sum vs annual revenue (exported years >= {START_YEAR}): {len(chk_in)} company-years, "
      f"{(chk_in['abs_pct_diff'] > 0.01).mean():.1%} differ by >1%")
mism = (chk_in[chk_in["abs_pct_diff"] > 0.01]
        .merge(company_meta[["cik", "ticker"]], on="cik")
        [["ticker", "fiscal_year", "q_revenue_sum", "revenue", "abs_pct_diff"]]
        .sort_values("abs_pct_diff", ascending=False))
print("Mismatched company-years in the exported range (flagged via dq_quarter_sum_mismatch):")
print(mism.to_string(index=False) if len(mism) else "  none")

# (b) sanity flags kept in the dataset instead of silently dropping rows
panel["dq_negative_revenue"] = panel["revenue"] < 0
panel["dq_missing_revenue"] = panel["revenue"].isna()
panel["dq_period_length_unusual"] = ~panel["period_days"].between(Q_MIN, Q_MAX)
print(panel[["dq_negative_revenue", "dq_missing_revenue", "dq_period_length_unusual",
             "dq_quarter_looks_annual", "dq_quarter_sum_mismatch"]].sum())

Quarter-sum vs annual revenue (all years incl. early XBRL): 432 company-years, 6.5% differ by >1%
Quarter-sum vs annual revenue (exported years >= 2014): 318 company-years, 6.6% differ by >1%
Mismatched company-years in the exported range (flagged via dq_quarter_sum_mismatch):
ticker  fiscal_year  q_revenue_sum      revenue  abs_pct_diff
    PG         2014   8.420100e+10 2.940000e+10      1.863980
   UAA         2014   2.439528e+09 2.082500e+09      0.171442
  MDXG         2016   2.450150e+08 2.217120e+08      0.105105
   JNJ         2023   9.302200e+10 8.515900e+10      0.092333
   MMM         2024   2.656200e+10 2.457500e+10      0.080855
   HON         2025   4.034000e+10 3.744200e+10      0.077400
   UAA         2021   4.801622e+09 4.474667e+09      0.073068
  MSFT         2016   8.532000e+10 9.115400e+10      0.064002
   MMM         2023   3.067000e+10 3.268100e+10      0.061534
   UAA         2016   4.056637e+09 3.825691e+09      0.060367
   UAA         2020   4.992650e+09 5.267

## 11. Export

In [17]:
def year_filter(df):
    keep = df["fiscal_year"] >= START_YEAR
    if END_YEAR is not None:
        keep &= df["fiscal_year"] <= END_YEAR
    return df[keep]


ID_COLS = ["cik", "ticker", "company", "sic", "sic_description", "group", "reference_note",
           "fiscal_year", "fiscal_quarter", "calendar_quarter", "period_start", "period_end", "period_days"]
RAW_COLS = DURATION_ITEMS + [c for c in INSTANT_ITEMS if c != "liabilities_and_equity"]
FEATURE_COLS = [
    "revenue_qoq_growth", "revenue_yoy_growth", "cogs_yoy_growth", "sga_yoy_growth", "net_income_yoy_growth",
    "ar_yoy_growth", "inventory_yoy_growth", "deferred_revenue_yoy_growth",
    "cogs_to_revenue", "sga_to_revenue", "rnd_to_revenue", "opex_to_revenue", "total_expense_to_revenue",
    "gross_margin", "operating_margin", "net_margin",
    "gross_margin_yoy_change", "operating_margin_yoy_change", "expense_ratio_yoy_change",
    "dso", "dio", "ar_growth_minus_revenue_growth", "inventory_growth_minus_revenue_growth",
    "sga_growth_minus_revenue_growth", "accruals_to_assets", "cfo_to_net_income", "leverage", "current_ratio",
]
Z_OUT = [f"z_{c}" for c in Z_METRICS] + [f"rz_{c}" for c in Z_METRICS] + ["n_abs_z_gt_3", "max_abs_z"]
PROV_COLS = ["revenue_latest", "revenue_restatement_pct", "n_items_restated", "n_bs_items_restated",
             "any_amended", "revenue_tag", "revenue_method", "revenue_tag_changed",
             "dq_negative_revenue", "dq_missing_revenue", "dq_period_length_unusual",
             "dq_quarter_looks_annual", "dq_quarter_sum_mismatch"]

quarterly_out = year_filter(panel)[ID_COLS + RAW_COLS + FEATURE_COLS + Z_OUT + PROV_COLS]
quarterly_out = quarterly_out.sort_values(["ticker", "period_end"])

A_ID = ["cik", "ticker", "company", "sic", "sic_description", "group", "reference_note", "fiscal_year", "period_end"]
A_FEAT = ["revenue_yoy_growth", "gross_margin", "operating_margin", "net_margin", "total_expense_to_revenue", "dso",
          "beneish_dsri", "beneish_gmi", "beneish_aqi", "beneish_sgi", "beneish_depi", "beneish_sgai",
          "beneish_lvgi", "beneish_tata", "beneish_m_score", "beneish_flag", "n_items_restated"]
annual_out = year_filter(annual)[A_ID + RAW_COLS + A_FEAT].sort_values(["ticker", "period_end"])

facts_out = facts.merge(company_meta[["cik", "ticker"]], on="cik", how="left")
facts_out = facts_out[["cik", "ticker", "item", "tag", "start", "end", "duration_days", "val", "first_val",
                       "restatement_pct", "restated", "n_reports", "first_filed", "last_filed", "last_form",
                       "last_accn", "amended"]]

coverage = (quarterly_out.groupby(["ticker", "company", "group"])
            .agg(first_quarter=("period_end", "min"), last_quarter=("period_end", "max"),
                 n_quarters=("period_end", "size"),
                 revenue_coverage=("revenue", lambda s: s.notna().mean()),
                 cfo_coverage=("cfo", lambda s: s.notna().mean()),
                 ar_coverage=("accounts_receivable", lambda s: s.notna().mean()),
                 quarters_with_restatement=("n_items_restated", lambda s: (s > 0).sum()))
            .reset_index())

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
quarterly_out.to_csv(OUTPUT_DIR / "quarterly_features.csv", index=False)
annual_out.to_csv(OUTPUT_DIR / "annual_features.csv", index=False)
facts_out.to_csv(OUTPUT_DIR / "facts_long.csv", index=False)
coverage.to_csv(OUTPUT_DIR / "coverage_report.csv", index=False)
print(f"Files written to: {OUTPUT_DIR.resolve()}")
print(f"quarterly_features.csv : {quarterly_out.shape}")
print(f"annual_features.csv    : {annual_out.shape}")
print(f"facts_long.csv         : {facts_out.shape}")
coverage

Files written to: C:\Users\Asus\Downloads\h1\data\processed
quarterly_features.csv : (1334, 94)
annual_features.csv    : (323, 48)
facts_long.csv         : (47514, 17)


,ticker,company,group,first_quarter,last_quarter,n_quarters,revenue_coverage,cfo_coverage,ar_coverage,quarters_with_restatement
0,AAPL,Apple Inc.,control,2013-12-28,2026-06-27,51,1.00000,1.00,1.000000,4
1,ADBE,ADOBE INC.,control,2014-02-28,2026-08-28,51,1.00000,1.00,1.000000,4
2,CAT,CATERPILLAR INC,control,2014-03-31,2026-06-30,50,1.00000,1.00,1.000000,10
3,COST,COSTCO WHOLESALE CORP /NEW,control,2013-11-24,2026-05-10,51,1.00000,1.00,1.000000,2
4,CSCO,"CISCO SYSTEMS, INC.",control,2013-10-26,2026-07-25,52,1.00000,1.00,1.000000,0
5,HD,"HOME DEPOT, INC.",control,2013-05-05,2026-08-02,54,1.00000,1.00,1.000000,14
6,HON,HONEYWELL INTERNATIONAL INC,control,2014-03-31,2026-06-30,50,1.00000,1.00,1.000000,25
7,INTC,INTEL CORP,control,2014-03-29,2026-06-27,50,1.00000,1.00,1.000000,14
8,JNJ,JOHNSON & JOHNSON,control,2014-03-30,2026-06-28,50,1.00000,1.00,1.000000,16
9,KHC,Kraft Heinz Co,reference,2014-03-30,2026-06-27,50,1.00000,0.96,0.920000,14


## 12. Quick look: largest trailing-Z deviations
A preview of what Deliverable 2 will model. High values are *candidates for review*, not evidence of fraud:
acquisitions, divestitures, accounting-standard changes and tag switches all create legitimate jumps.

In [18]:
(quarterly_out[~quarterly_out["revenue_tag_changed"] & ~quarterly_out["dq_quarter_sum_mismatch"]]
 .sort_values("max_abs_z", ascending=False)
 [["ticker", "fiscal_year", "fiscal_quarter", "revenue", "revenue_yoy_growth", "total_expense_to_revenue",
   "dso", "accruals_to_assets", "max_abs_z", "n_abs_z_gt_3", "n_items_restated"]]
 .head(20))

,ticker,fiscal_year,fiscal_quarter,revenue,revenue_yoy_growth,total_expense_to_revenue,dso,accruals_to_assets,max_abs_z,n_abs_z_gt_3,n_items_restated
1813,KHC,2018,4,6.891000e+09,0.007309,3.051371,28.114787,-0.127631,60.626235,3,0
1352,CSCO,2018,2,1.188700e+10,0.026511,0.741482,30.338437,-0.098064,30.028331,2,0
1645,ORCL,2022,2,1.036000e+10,0.057143,1.079537,39.193243,0.021190,23.400314,3,0
1621,ORCL,2016,2,8.993000e+09,-0.063034,0.671411,40.030691,0.015690,21.954749,1,1
699,PG,2019,4,1.709400e+10,0.035812,1.303732,26.356675,-0.079335,21.853337,3,0
1800,KHC,2015,3,6.120000e+09,1.359291,0.934804,23.984150,-0.005803,17.085219,3,0
921,NKE,2020,4,6.313000e+09,-0.380106,NaN,40.061460,-0.027414,15.969841,3,0
1839,KHC,2025,2,6.352000e+09,-0.019148,2.255353,33.580605,-0.105124,14.662558,3,0
1304,ADBE,2024,1,5.182000e+09,0.113212,0.824971,36.122540,-0.018930,14.081688,2,0
1196,MSFT,2015,4,2.218000e+10,-0.051407,1.092561,73.472858,-0.056735,12.724346,2,1
